# FD001 training-only exploration
Chua Shanwei, 35393343. This optional notebook audits training histories without reading official test labels. Install the project first. The command-line workflow is the authoritative reproducible experiment.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from jet_rul.data import read_cmapss, SENSORS
from jet_rul.features import build_features, attach_targets
root = Path.cwd() if (Path.cwd() / 'configs').exists() else Path.cwd().parent
data_dir = root / 'data/raw'
if not (data_dir / 'train_FD001.txt').exists():
    data_dir = root.parent / 'CMAPSSData'
train = read_cmapss(data_dir / 'train_FD001.txt')
print(train.shape, train.engine_id.nunique())
print(train[SENSORS].describe())

In [ ]:
for engine in [1, 2, 3]:
    history = train.loc[train.engine_id == engine]
    plt.plot(history.cycle, history.s4.rolling(20).mean(), label=f'Engine {engine}')
plt.xlabel('Operational cycle')
plt.ylabel('Trailing mean T50 (degrees Rankine)')
plt.legend()
plt.show()

In [ ]:
features = attach_targets(build_features(train, window=30), train)
print(features.head())
assert (features.target >= 0).all()
assert features.groupby('engine_id').target.last().eq(0).all()
print('Training endpoints correctly have zero RUL. Never apply this rule to censored test data.')

## Interpretation
Different failure ages and sensor histories motivate prognostics beyond age alone. Correlation does not prove a causal wear mechanism. Continue with the fixed engine-partition workflow in the README; do not choose predictors or settings from official test performance.